In [5]:
#%%imports
import h5py
import importlib
import numpy as np
from typing import Tuple, List

from lust_codesnippets_py.io import hdf5

importlib.reload(hdf5)

<module 'lust_codesnippets_py.io.hdf5' from '/home/lukas/github/LuStCodeSnippets/lust_codesnippets_py/io/hdf5.py'>

In [6]:
def gen_data(n:int, sigma_nu:float=0.01) -> Tuple[List[np.ndarray],List[float],List[np.ndarray]]:
    """
        - helper function to generate some data
        - will generate `n` observations for the lc with gaussian noise of level of `sigma_nu`
            - errors are assigned as `2*np.abs(flux_e)`

        Parameters
        ----------
            - `n`
                - `int`
                - number of observations to generate for the pseudo-lc
            - `sigma_nu`
                - `float`, optional
                - amplitude of the noise added to the pseudo-lc

        Raises
        ------

        Returns
        -------
            - `images`
                - `List[np.ndarray]
                - 3 images
            - `lc_params`
                - `List[float]`
                - params used to generate the pseudo-lc
            - `lc`
                - `List[np.ndarray]`
                - pseudo-obvservations of the lc

        Dependencies
        ------------
            - `numpy`

        Comments
        --------
    """
    im1 = np.random.randn(10,10)
    im2 = np.arange(50).reshape(5,10)
    im3 = np.random.randint(0, 255, (50, 50, 3), dtype=np.uint8)   #rgb image with random colors    
    images = [im1,im2,im3]

    #get lc
    ##parameters
    unique_bands = list("ugrizy")
    mu1, sigma1 = 40, 20
    mu2, sigma2 = 20, 10 
    lc_params = [mu1, mu2, sigma1, sigma2, unique_bands]
    
    ##measurements
    measid  = np.arange(0,n,1)
    time    = np.linspace(0,1.5*max(mu1, mu2),n) + 0.05*np.random.randn(n)
    flux    = np.exp(-(time - mu1)**2/sigma1**2) + np.exp(-(time - mu2)**2/sigma2**2)
    flux_e  = sigma_nu*np.random.randn(n)         #errorbars and noise
    band    = np.random.choice(unique_bands, size=n)
    lc = [measid,time,flux+flux_e,2*np.abs(flux_e),band]
    
    return images, lc_params, lc

In [ ]:
def makehdf5(filepath:str):
    """
        - function to create an hdf5 file
    """
    
    #get data
    images, lc_params, lc = gen_data(100, 0.01)
    im1, im2, im3 = images
    mu1, mu2, sigma1, sigma2, unique_bands = lc_params
    measid, time, flux, flux_e, band = lc 
    
    #create file
    f = h5py.File(filepath, "w")
    
    ##group for images
    grp1 = f.create_group("images")
    ds11 = grp1.create_dataset("observation",   data=im1, dtype=float)
    ds12 = grp1.create_dataset("gradient",      data=im2, dtype=float)
    ds13 = grp1.create_dataset("colors",        data=im3, dtype=float)
    ##group for LC
    grp2 = f.create_group("lightcurve")
    ds21 = grp2.create_dataset("measid",        data=measid, dtype=int)
    ds22 = grp2.create_dataset("time",          data=time, dtype=float)
    ds23 = grp2.create_dataset("flux",          data=flux, dtype=float)
    ds24 = grp2.create_dataset("flux_e",        data=flux_e, dtype=float)
    ds25 = grp2.create_dataset("band",          data=band.astype("S1"))     #strings need to be specified explicitly
    ###add metadata
    grp2.attrs["mu1"] = mu1
    grp2.attrs["mu2"] = mu2
    grp2.attrs["sigma1"] = sigma1
    grp2.attrs["sigma2"] = sigma2
    grp2.attrs["unique_bands"] = "".join(unique_bands)

    f.close()

    return


In [10]:
makehdf5("../../data/temp.h5")
hdf5.tree("../../data/temp.h5")
f = h5py.File("../../data/temp.h5", "r")
hdf5.tree(f)
f.close()

tree: <HDF5 file "temp.h5" (mode r)>
|--images          (                             )
|  |--colors       ((50, 50, 3)    , float64   , )
|  |--gradient     ((5, 10)        , float64   , )
|  `--observation  ((10, 10)       , float64   , )
`--lightcurve      (                             mu1:40, mu2:20, sigma1:20, sigma2:10, unique_bands:ugrizy)
   |--band         ((100,)         , |S1       , )
   |--flux         ((100,)         , float64   , )
   |--flux_e       ((100,)         , float64   , )
   |--measid       ((100,)         , int64     , )
   `--time         ((100,)         , float64   , )
tree: <HDF5 file "temp.h5" (mode r)>
|--images          (                             )
|  |--colors       ((50, 50, 3)    , float64   , )
|  |--gradient     ((5, 10)        , float64   , )
|  `--observation  ((10, 10)       , float64   , )
`--lightcurve      (                             mu1:40, mu2:20, sigma1:20, sigma2:10, unique_bands:ugrizy)
   |--band         ((100,)         , |S1       